In [5]:
from metric import table_datapoints_precision_recall
from pathlib import Path
from annotation2markdown import json_to_markdown
import statistics

def compute_chart_metrics(path_to_preds, path_to_gt):
    """
    Calcola le medie delle metriche per classe di difficoltà e globalmente.
    Gestisce predizioni sia in formato .json che .txt.
    """
    results_by_diff = {}
    global_accum = {'p': [], 'r': [], 'f1': []}

    for diff_class in path_to_gt.iterdir():
        if not diff_class.is_dir():
            continue
            
        class_name = diff_class.name
        results_by_diff[class_name] = {'p': [], 'r': [], 'f1': []}
        
        for gt_file in diff_class.glob("*.json"):
            # Cerca la predizione corrispondente (può essere .json o .txt)
            pred_path_json = path_to_preds / class_name / gt_file.name
            pred_path_txt = pred_path_json.with_suffix('.txt')
            
            # Determinazione del formato e caricamento per predizione formato json gemini o txt DePlot
            if pred_path_json.exists():
                pred_table = json_to_markdown(pred_path_json)
            elif pred_path_txt.exists():
                with open(pred_path_txt, 'r', encoding='utf-8') as f:
                    content = f.read()
                    pred_table = [content, content]
            else:
                continue

            gt_table = json_to_markdown(gt_file)
            
            # Calcolo metriche [cite: 15]
            metrics = table_datapoints_precision_recall(
                [[pred_table[0]], [pred_table[1]]], 
                [gt_table[0], gt_table[1]]
            )

            p = metrics['table_datapoints_precision']
            r = metrics['table_datapoints_recall']
            f1 = metrics['table_datapoints_f1']

            results_by_diff[class_name]['p'].append(p)
            results_by_diff[class_name]['r'].append(r)
            results_by_diff[class_name]['f1'].append(f1)

            global_accum['p'].append(p)
            global_accum['r'].append(r)
            global_accum['f1'].append(f1)

    # Aggregazione medie
    final_metrics = {'classes': {}, 'global': {}}
    for cn, m in results_by_diff.items():
        if m['f1']:
            final_metrics['classes'][cn] = {
                'avg_precision': statistics.mean(m['p']),
                'avg_recall': statistics.mean(m['r']),
                'avg_f1': statistics.mean(m['f1'])
            }

    if global_accum['f1']:
        final_metrics['global'] = {
            'avg_precision': statistics.mean(global_accum['p']),
            'avg_recall': statistics.mean(global_accum['r']),
            'avg_f1': statistics.mean(global_accum['f1'])
        }
        
    return final_metrics

In [30]:
chart_types = ["pie", "line", "bar"]
base_pred_path = Path("../predictions/Gemini/pmc/")
base_gt_path = Path("../data_groundtruth/pmc/")

for chart in chart_types:
    print(f"\n=== ANALISI CHART: {chart.upper()} ===")
    
    res = compute_chart_metrics(base_pred_path / chart, base_gt_path / chart)
    
    for diff, m in res['classes'].items():
        print(f"[{diff}] F1: {m['avg_f1']:.2f}% | P: {m['avg_precision']:.2f}% | R: {m['avg_recall']:.2f}%")
    
    if res['global']:
        g = res['global']
        print(f"GLOBAL -> F1: {g['avg_f1']:.2f}% | P: {g['avg_precision']:.2f}% | R: {g['avg_recall']:.2f}%")


=== ANALISI CHART: PIE ===
[hard] F1: 85.64% | P: 85.64% | R: 85.64%
[medium] F1: 70.72% | P: 83.47% | R: 68.97%
[easy] F1: 100.00% | P: 100.00% | R: 100.00%
GLOBAL -> F1: 85.45% | P: 89.70% | R: 84.87%

=== ANALISI CHART: LINE ===
[hard] F1: 100.00% | P: 100.00% | R: 100.00%
[medium] F1: 100.00% | P: 100.00% | R: 100.00%
[easy] F1: 100.00% | P: 100.00% | R: 100.00%
GLOBAL -> F1: 100.00% | P: 100.00% | R: 100.00%

=== ANALISI CHART: BAR ===
[hard] F1: 82.54% | P: 82.54% | R: 82.54%
[medium] F1: 94.50% | P: 94.50% | R: 94.50%
[easy] F1: 99.87% | P: 99.87% | R: 99.87%
GLOBAL -> F1: 92.30% | P: 92.30% | R: 92.30%


In [31]:
chart_types = ["pie", "line", "bar"] 
base_pred_path = Path("../predictions/DePlot/pmc/")
base_gt_path = Path("../data_groundtruth/pmc/")

for chart in chart_types:
    print(f"\n=== ANALISI CHART: {chart.upper()} ===")
    
    res = compute_chart_metrics(base_pred_path / chart, base_gt_path / chart)
    
    for diff, m in res['classes'].items():
        print(f"[{diff}] F1: {m['avg_f1']:.2f}% | P: {m['avg_precision']:.2f}% | R: {m['avg_recall']:.2f}%")
    
    if res['global']:
        g = res['global']
        print(f"GLOBAL -> F1: {g['avg_f1']:.2f}% | P: {g['avg_precision']:.2f}% | R: {g['avg_recall']:.2f}%")


=== ANALISI CHART: PIE ===
[hard] F1: 25.73% | P: 23.16% | R: 28.95%
[medium] F1: 12.49% | P: 11.87% | R: 13.32%
[easy] F1: 10.61% | P: 14.58% | R: 8.33%
GLOBAL -> F1: 16.27% | P: 16.54% | R: 16.87%

=== ANALISI CHART: LINE ===
[hard] F1: 8.45% | P: 5.73% | R: 23.44%
[medium] F1: 7.41% | P: 6.36% | R: 9.38%
[easy] F1: 25.73% | P: 25.73% | R: 25.73%
GLOBAL -> F1: 14.54% | P: 13.47% | R: 19.03%

=== ANALISI CHART: BAR ===
[hard] F1: 0.00% | P: 0.00% | R: 0.00%
[medium] F1: 16.41% | P: 16.98% | R: 15.90%
[easy] F1: 21.82% | P: 21.82% | R: 21.82%
GLOBAL -> F1: 12.74% | P: 12.93% | R: 12.57%


# RMM su grafici genreati sinteticamente

In [1]:
from metric import table_datapoints_precision_recall
from pathlib import Path
from annotation2markdown import json_to_markdown
import statistics

def compute_chart_metrics_global(path_to_preds, path_to_gt):
    """
    Calcola le medie delle metriche globalmente per tutti i file.
    Gestisce predizioni sia in formato .json che .txt.
    """
    global_accum = {'p': [], 'r': [], 'f1': []}

    # Itera direttamente sui file .json nella cartella ground truth
    for gt_file in path_to_gt.glob("*.json"):
        
        # Cerca la predizione corrispondente nella root di path_to_preds
        pred_path_json = path_to_preds / gt_file.name
        pred_path_txt = pred_path_json.with_suffix('.txt')
        
        # Determinazione del formato e caricamento per predizione formato json gemini o txt DePlot
        if pred_path_json.exists():
            pred_table = json_to_markdown(pred_path_json)
        elif pred_path_txt.exists():
            with open(pred_path_txt, 'r', encoding='utf-8') as f:
                content = f.read()
                pred_table = [content, content]
        else:
            # Salta se non trova la predizione in nessun formato
            continue

        gt_table = json_to_markdown(gt_file)
        
        # Calcolo metriche
        metrics = table_datapoints_precision_recall(
            [[pred_table[0]], [pred_table[1]]], 
            [gt_table[0], gt_table[1]]
        )

        global_accum['p'].append(metrics['table_datapoints_precision'])
        global_accum['r'].append(metrics['table_datapoints_recall'])
        global_accum['f1'].append(metrics['table_datapoints_f1'])

    # Aggregazione medie globali
    final_metrics = {}
    if global_accum['f1']:
        final_metrics = {
            'avg_precision': statistics.mean(global_accum['p']),
            'avg_recall': statistics.mean(global_accum['r']),
            'avg_f1': statistics.mean(global_accum['f1'])
        }
        
    return final_metrics

In [4]:
chart_types = ["bar", "line", "pie"] 
base_pred_path = Path("../predictions/Gemini/synthetic/")
base_pred_path = Path("../predictions/Gemini/synthetic/")
base_gt_path = Path("../data_groundtruth/synthetic/")

for chart in chart_types:
    print(f"\n=== ANALISI CHART: {chart.upper()} ===")
    
    res = compute_chart_metrics_global(base_pred_path / chart, base_gt_path / chart)
    print(res)
    print(f"GLOBAL -> F1: {res['avg_f1']:.2f}% | P: {res['avg_precision']:.2f}% | R: {res['avg_recall']:.2f}%")


=== ANALISI CHART: BAR ===
{'avg_precision': 71.37191878379899, 'avg_recall': 69.91185104494063, 'avg_f1': 70.41905446806012}
GLOBAL -> F1: 70.42% | P: 71.37% | R: 69.91%

=== ANALISI CHART: LINE ===
{'avg_precision': 53.45491549064706, 'avg_recall': 61.31664135077383, 'avg_f1': 55.621968427838354}
GLOBAL -> F1: 55.62% | P: 53.45% | R: 61.32%

=== ANALISI CHART: PIE ===
{'avg_precision': 36.285490569277215, 'avg_recall': 33.81586003826966, 'avg_f1': 34.232521624486296}
GLOBAL -> F1: 34.23% | P: 36.29% | R: 33.82%


In [ ]:
chart_types = ["bar", "line", "pie"] 
base_pred_path = Path("../predictions/DePlot/synthetic/")
base_pred_path = Path("../predictions/DePlot/synthetic/")
base_gt_path = Path("../data_groundtruth/synthetic/")

for chart in chart_types:
    print(f"\n=== ANALISI CHART: {chart.upper()} ===")
    
    res = compute_chart_metrics_global(base_pred_path / chart, base_gt_path / chart)
    print(res)
    print(f"GLOBAL -> F1: {res['avg_f1']:.2f}% | P: {res['avg_precision']:.2f}% | R: {res['avg_recall']:.2f}%")


=== ANALISI CHART: BAR ===
{'avg_precision': 50.38538205869097, 'avg_recall': 50.09266605744144, 'avg_f1': 49.850971860615445}
GLOBAL -> F1: 49.85% | P: 50.39% | R: 50.09%

=== ANALISI CHART: LINE ===
{'avg_precision': 28.752597796984695, 'avg_recall': 34.29655027344183, 'avg_f1': 30.453256033121125}
GLOBAL -> F1: 30.45% | P: 28.75% | R: 34.30%

=== ANALISI CHART: PIE ===
{'avg_precision': 11.17870085196474, 'avg_recall': 19.486337747102674, 'avg_f1': 12.61010565782647}
GLOBAL -> F1: 12.61% | P: 11.18% | R: 19.49%


: 